# 02 — Làm sạch (baseline), chia train/test, tăng cường, xuất 2 bộ dữ liệu

Luồng: **raw -> audit SNR (chỉ để biết, KHÔNG loại mẫu) -> chia train/test
(stratify theo nồng độ) -> augment CHỈ trên train -> xuất 2 bộ dữ liệu:**

1. **Bộ phổ thô** (baseline-corrected): `train_spectra_original.csv`,
   `train_spectra_augmented.csv`, `test_spectra.csv`
2. **Bộ đặc trưng đỉnh** (sau peak detection, từ đúng các phổ ở bộ 1):
   `peak_features_train.csv`, `peak_features_test.csv`

Xuất song song 2 bộ để so sánh hiệu quả mô hình khi học trực tiếp từ phổ
thô (dạng bộ 1) so với khi học từ đặc trưng đỉnh đã trích xuất (dạng bộ 2).

Toàn bộ tham số (`lam`, `window`, `k`) nạp từ `data/processed/chosen_params.json`
(xuất ra từ `00_parameter_selection.ipynb`), không hard-code lại ở đây.

In [ ]:
import sys, os, json
sys.path.insert(0, '../src')  # chỉnh lại nếu vị trí src/ khác

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from raman_processing import (baseline_correction, smooth_spectrum, detect_peaks,
                               extract_peak_features, adaptive_prominence, calc_snr_std)
from data_cleaning import build_label_table, build_snr_report, flag_low_snr_samples, summarize_flags
from augmentation import augment_dataset

os.makedirs('../data/processed', exist_ok=True)

with open('../data/processed/chosen_params.json') as f:
    params = json.load(f)
print('Tham số đã chọn (từ 00_parameter_selection.ipynb):')
print(json.dumps(params, indent=2))

AIRPLS_LAM = params['airpls_lam']
SMOOTH_WINDOW = params['smooth_window']
PROMINENCE_K = params['prominence_k']
PEAK_DISTANCE = params.get('peak_distance', 1)
PEAK_WIDTH = params.get('peak_width', 0.5)

## 1. Đọc dữ liệu gốc

In [ ]:
df_raw = pd.read_excel('../data/raw/Ethanol_Methanol.xlsx')
x_full = df_raw['Raman Shift (cm-1)'].values
mask = ~np.isnan(x_full)
x = x_full[mask]
df = df_raw.loc[mask].reset_index(drop=True)

sample_cols = [c for c in df.columns if c != 'Raman Shift (cm-1)']
print(f'{len(sample_cols)} mẫu, {len(x)} điểm/phổ')

## 2. Làm sạch: baseline correction cho toàn bộ mẫu (KHÔNG loại mẫu nào)

Audit SNR vẫn chạy để bạn biết mẫu nào yếu (vd các mẫu `_c`), nhưng chỉ để
tham khảo/ghi chú trong `snr_report.csv` — không tự động loại khỏi pipeline.

In [ ]:
snr_report = build_snr_report(df, x, sample_cols=sample_cols,
                               baseline_method='airpls', lam=AIRPLS_LAM)
flagged = flag_low_snr_samples(snr_report, ratio_threshold=0.3)
_ = summarize_flags(flagged)  # chỉ in ra để bạn biết, không dùng để loại mẫu

flagged.to_csv('../data/processed/snr_report.csv', index=False)
print(f'\nĐã lưu snr_report.csv (tham khảo) -- giữ nguyên toàn bộ {len(sample_cols)} mẫu')

### ⚠️ Sửa quan trọng: chuẩn hoá cường độ (khắc phục batch effect)

Kiểm tra dữ liệu cho thấy cường độ tuyệt đối giữa các lần đo lệch nhau
rất nhiều dù CÙNG nồng độ (vd `E90_a` max≈7191 vs `E90_c` max≈702, lệch
~10 lần) -- nhiều khả năng do laser power / thời gian tích phân khác
nhau giữa các lần đo, KHÔNG phải do nồng độ. Nếu để nguyên, model sẽ học
nhầm "cường độ tuyệt đối" thay vì tín hiệu nồng độ thật, dẫn tới R² âm
khi test rơi vào lô đo khác với train (xem `diagnose_model_issues.py`).

**Sửa:** thêm bước chuẩn hoá L2-norm cho từng phổ ngay sau
`baseline_correction()`, trước khi augment/export. Chuẩn hoá theo từng
phổ độc lập nên không có rủi ro rò rỉ thông tin giữa train/test.

In [ ]:
def normalize_spectrum(spectrum, method='l2'):
    """
    Loại bỏ chênh lệch scale cường độ giữa các lần đo (laser power/thời
    gian tích phân khác nhau) -- áp dụng SAU baseline_correction(), TRƯỚC
    augmentation/PCA/model. Xem norm_note ở trên để biết lý do cần bước này.

    method: 'l2' (mặc định, khuyến nghị) | 'max' | 'area'
    """
    spectrum = np.asarray(spectrum, dtype=float).ravel()
    if method == 'l2':
        denom = np.linalg.norm(spectrum)
    elif method == 'max':
        denom = spectrum.max()
    elif method == 'area':
        denom = spectrum.sum()
    else:
        raise ValueError(f"Unknown method: {method}")
    return spectrum / (denom + 1e-9)


# Baseline-correct + chuẩn hoá cường độ toàn bộ mẫu, giữ nguyên hết -- đây
# là "dữ liệu sạch" dùng cho các bước sau
df_clean = pd.DataFrame({
    col: normalize_spectrum(
        baseline_correction(df[col].values.astype(float), method='airpls', lam=AIRPLS_LAM),
        method='l2')
    for col in sample_cols
})
labels_all = build_label_table(sample_cols)
print('Đã baseline-correct + chuẩn hoá (l2)', df_clean.shape[1], 'mẫu (giữ nguyên toàn bộ)')

## 3. Chia train / test (stratify theo nồng độ ĐÃ BIN, theo PHỔ GỐC)

Chia trước khi augment. Mẫu có `concentration=None` (EMX, EMX_b) đưa hết vào
train vì không thể stratify theo nồng độ số.

**Sửa quan trọng:** stratify theo nồng độ CHÍNH XÁC (bản cũ) khiến nhiều
nhóm (Ethanol/Methanol nguyên chất, E5, E10) chỉ có 1 mẫu -> điều kiện
`min >= 2` fail ÂM THẦM, code rơi về random split mà không cảnh báo, mất
tác dụng stratify hoàn toàn dù nhìn code tưởng vẫn hoạt động. Bản mới bin
nồng độ theo nhiều mức từ chặt đến lỏng, tự chọn mức chặt nhất vẫn đảm
bảo mọi nhóm strat >= 2 mẫu, và LUÔN in cảnh báo rõ nếu buộc phải rơi về
random split.

In [ ]:
has_conc = labels_all['concentration'].notna()
df_with_conc = labels_all[has_conc].copy()
df_no_conc = labels_all[~has_conc].copy()

# --- Stratify theo nồng độ đã BIN, không dùng giá trị chính xác ---
# Lý do: với bộ dữ liệu này, nhiều mức nồng độ (Ethanol/Methanol nguyên
# chất, E5, E10) chỉ có 1 mẫu duy nhất -> điều kiện cũ
# (strat_key.value_counts().min() >= 2) fail ÂM THẦM, khiến code rơi về
# random split (stratify=None) mà KHÔNG cảnh báo gì -- tưởng vẫn stratify
# nhưng thực chất không hề. Ở đây thử nhiều mức bin/tổ hợp key từ chặt tới
# lỏng, chọn mức chặt nhất vẫn đảm bảo stratify hợp lệ, và LUÔN in cảnh
# báo rõ nếu phải rơi về random split.
#
# Lưu ý: theo data_cleaning.parse_sample_label(), Ethanol/Methanol nguyên
# chất đều thuộc series='E' (không phải nhãn series riêng) -- nên chỉ cần
# clip nồng độ 100 -> 99 trước khi bin là đủ để chúng nhập chung nhóm với
# E80/E90 cùng series, không cần gộp series thủ công.

def build_strat_key(df, bin_width, use_series):
    # clip 100 -> 99 trước khi bin: Ethanol nguyên chất (concentration=100)
    # sẽ nhập chung vào bin cao nhất (cùng series 'E' với E80/E90) thay vì
    # tự thành 1 nhóm riêng chỉ có 1 mẫu
    conc = df['concentration'].clip(upper=99)
    conc_bin = (conc // bin_width) * bin_width
    if use_series:
        return df['series'].astype(str) + '_' + conc_bin.astype(str)
    return conc_bin.astype(str)

# thử từ chặt (bin hẹp, có phân biệt series) đến lỏng (bin rộng, gộp series)
candidate_configs = [
    (10, True), (20, True), (25, True),
    (20, False), (25, False), (50, False), (101, False),
]

strat_key = None
for bin_width, use_series in candidate_configs:
    candidate = build_strat_key(df_with_conc, bin_width, use_series)
    if candidate.value_counts().min() >= 2:
        strat_key = candidate
        print(f'✅ Stratify hợp lệ với bin_width={bin_width}, use_series={use_series} '
              f'-- nhóm nhỏ nhất có {candidate.value_counts().min()} mẫu')
        break

if strat_key is None:
    print('⚠️  CẢNH BÁO: không tìm được cách bin nồng độ nào đảm bảo mọi nhóm '
          '>= 2 mẫu -- rơi về RANDOM SPLIT THUẦN TÚY (không stratify theo '
          'nồng độ). Cân nhắc thu thập thêm mẫu ở các nồng độ hiếm để khắc '
          'phục triệt để.')
else:
    print('\nPhân bố các nhóm strat được dùng:')
    print(strat_key.sort_index().value_counts().sort_index())

train_raw, test_raw = train_test_split(
    df_with_conc['raw'],
    test_size=0.2,
    random_state=42,
    stratify=strat_key,
)

train_raw = pd.concat([train_raw, df_no_conc['raw']], ignore_index=True)

split_df = pd.DataFrame({
    'raw': list(train_raw) + list(test_raw),
    'split': ['train'] * len(train_raw) + ['test'] * len(test_raw),
})
split_df.to_csv('../data/processed/split_ids.csv', index=False)
print(f'\nTrain: {len(train_raw)} mẫu | Test: {len(test_raw)} mẫu')
print(split_df.groupby('split').size())


## 4. Tăng cường dữ liệu — CHỈ áp dụng cho tập train

Test set giữ nguyên phổ thật, không augment.

In [ ]:
N_AUGMENTATIONS = 100  # chỉnh theo mục tiêu số phổ/lớp

train_cols = split_df.loc[split_df['split'] == 'train', 'raw'].tolist()
test_cols = split_df.loc[split_df['split'] == 'test', 'raw'].tolist()

aug_df, source_map = augment_dataset(
    df_clean, x, sample_cols=train_cols,
    n_augmentations=N_AUGMENTATIONS, seed=42,
)
print('Số phổ tổng hợp:', aug_df.shape[1])

## 5. Xuất bộ 1 — phổ thô (baseline-corrected) train/test

In [ ]:
train_raw_spectra = df_clean[train_cols].copy()
test_spectra = df_clean[test_cols].copy()

x_series = pd.Series(x, name='Raman Shift (cm-1)')

pd.concat([x_series, train_raw_spectra], axis=1).to_csv(
    '../data/processed/train_spectra_original.csv', index=False)
pd.concat([x_series, aug_df], axis=1).to_csv(
    '../data/processed/train_spectra_augmented.csv', index=False)
pd.concat([x_series, test_spectra], axis=1).to_csv(
    '../data/processed/test_spectra.csv', index=False)

pd.Series(source_map, name='source_sample').rename_axis('augmented_col').to_csv(
    '../data/processed/augmented_source_map.csv')

print('Đã lưu bộ 1 (phổ thô):')
print(' - train_spectra_original.csv  ', train_raw_spectra.shape)
print(' - train_spectra_augmented.csv ', aug_df.shape)
print(' - test_spectra.csv            ', test_spectra.shape)

## 6. Xuất bộ 2 — đặc trưng đỉnh (peak features) train/test

Chạy peak detection (smoothing + adaptive threshold + detect_peaks +
extract_peak_features) trên ĐÚNG các phổ vừa xuất ở bộ 1 (train gốc, train
augmented, test) -- dùng chung tham số `SMOOTH_WINDOW`, `PROMINENCE_K`, `PEAK_DISTANCE`,
`PEAK_WIDTH` đã chọn ở `00_parameter_selection.ipynb`.

Mỗi phổ có thể fit lỗi (đặc biệt với 2700 phổ augmented) -- bọc try/except
để 1 lỗi không làm chết cả vòng lặp, log lại số lỗi để bạn biết mức độ.

In [ ]:
def build_peak_feature_table(spectra_df, x, window, prominence_k, distance, width, tag):
    """spectra_df: mỗi cột là 1 phổ ĐÃ baseline-corrected. distance/width lấy
    từ chosen_params.json (00_parameter_selection.ipynb), không hard-code.
    Trả về DataFrame đặc trưng đỉnh (center/area/height/fwhm) gộp từ mọi
    cột, kèm cột 'sample'."""
    all_features = []
    n_failed = 0
    for col in spectra_df.columns:
        corrected = spectra_df[col].values.astype(float)
        smoothed = smooth_spectrum(corrected, window=window, polyorder=3)
        thr = adaptive_prominence(corrected, x, k=prominence_k)
        peaks = detect_peaks(smoothed, x=x, prominence=thr, distance=distance, width=width)
        try:
            features = extract_peak_features(smoothed, x, peaks, sigma_max=30)
        except Exception as e:
            n_failed += 1
            features = []
        for f in features:
            f['sample'] = col
            all_features.append(f)
    print(f'[{tag}] {spectra_df.shape[1]} phổ -> {len(all_features)} đỉnh '
          f'({n_failed} phổ fit lỗi, bỏ qua)')
    return pd.DataFrame(all_features)


peak_features_train_original = build_peak_feature_table(
    train_raw_spectra, x, SMOOTH_WINDOW, PROMINENCE_K, PEAK_DISTANCE, PEAK_WIDTH, 'train-original')
peak_features_train_augmented = build_peak_feature_table(
    aug_df, x, SMOOTH_WINDOW, PROMINENCE_K, PEAK_DISTANCE, PEAK_WIDTH, 'train-augmented')
peak_features_test = build_peak_feature_table(
    test_spectra, x, SMOOTH_WINDOW, PROMINENCE_K, PEAK_DISTANCE, PEAK_WIDTH, 'test')

peak_features_train = pd.concat(
    [peak_features_train_original.assign(source='original'),
     peak_features_train_augmented.assign(source='augmented')],
    ignore_index=True,
)
peak_features_test = peak_features_test.assign(source='original')

In [ ]:
peak_features_train.to_csv('../data/processed/peak_features_train.csv', index=False)
peak_features_test.to_csv('../data/processed/peak_features_test.csv', index=False)

print('Đã lưu bộ 2 (đặc trưng đỉnh):')
print(' - peak_features_train.csv ', peak_features_train.shape)
print(' - peak_features_test.csv  ', peak_features_test.shape)

## 7. Tổng hợp file đã xuất

| File | Nội dung |
|---|---|
| `train_spectra_original.csv` | Phổ train gốc, đã baseline-corrected, chưa augment |
| `train_spectra_augmented.csv` | Phổ train sau augment (100x/phổ gốc) |
| `test_spectra.csv` | Phổ test, đã baseline-corrected, KHÔNG augment |
| `peak_features_train.csv` | Đặc trưng đỉnh trích từ cả 2 file train ở trên (cột `source` phân biệt) |
| `peak_features_test.csv` | Đặc trưng đỉnh trích từ test |
| `snr_report.csv` | Báo cáo SNR toàn bộ mẫu (tham khảo, không dùng để loại mẫu) |
| `split_ids.csv` | Nhãn train/test theo từng phổ gốc |
| `augmented_source_map.csv` | Map từ tên cột augmented -> phổ gốc sinh ra nó |

So sánh mô hình học từ `train_spectra_*` (dạng phổ thô/full spectrum) với
mô hình học từ `peak_features_train.csv` (dạng đặc trưng đỉnh) ở bước
modeling tiếp theo.